# GLM-OCR

In [1]:
!pip install git+https://github.com/huggingface/transformers.git -q
!pip install pymupdf pillow -q

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 104.9 MB/s eta 0:00:00


In [2]:
import os
import time
import psutil
import torch
import fitz
import tempfile
from pathlib import Path
from PIL import Image
from transformers import AutoProcessor, AutoModelForImageTextToText

In [3]:
try:
    from google.colab import files
    uploaded = files.upload()
    pdf_path = list(uploaded.keys())[0]
    print("Using PDF:", pdf_path)

except ImportError:
    pdf_path = "data/benchmark/flexqube_arsredovisning_2022.pdf"

    print(f"Running locally, using: {pdf_path}")


Saving FlexQube.pdf to FlexQube.pdf
Using PDF: FlexQube.pdf


### Convert PDF pages to PIL images

In [4]:
def pdf_to_images(pdf_path, dpi=120):
    doc = fitz.open(pdf_path)
    images = []
    for i, page in enumerate(doc):
        mat = fitz.Matrix(dpi / 72, dpi / 72)
        pix = page.get_pixmap(matrix=mat, colorspace=fitz.csRGB)
        img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)

        max_dim = 1600
        if max(img.size) > max_dim:
            ratio = max_dim / max(img.size)
            img = img.resize((int(img.width * ratio), int(img.height * ratio)), Image.LANCZOS)

        images.append(img)
        print(f"  Page {i + 1}: {img.size}")
    doc.close()
    return images

pages = pdf_to_images(pdf_path)
print(f"Total: {len(pages)} pages")

  Page 1: (993, 1404)
  Page 2: (1600, 1131)
  Page 3: (1600, 1131)
MuPDF error: format error: No common ancestor in structure tree

  Page 4: (1600, 1131)
  Page 5: (1600, 1131)
  Page 6: (1600, 1131)
  Page 7: (1600, 1131)
  Page 8: (1600, 1131)
  Page 9: (1600, 1131)
  Page 10: (1600, 1131)
  Page 11: (1600, 1131)
  Page 12: (1600, 1131)
  Page 13: (1600, 1131)
  Page 14: (1600, 1131)
  Page 15: (1600, 1131)
  Page 16: (1600, 1131)
  Page 17: (1600, 1131)
  Page 18: (1600, 1131)
  Page 19: (1600, 1131)
  Page 20: (1600, 1131)
  Page 21: (1600, 1131)
  Page 22: (1600, 1131)
  Page 23: (1600, 1131)
  Page 24: (1600, 1131)
  Page 25: (1600, 1131)
  Page 26: (1600, 1131)
  Page 27: (1600, 1131)
  Page 28: (1600, 1131)
  Page 29: (1600, 1131)
  Page 30: (1600, 1131)
  Page 31: (1600, 1131)
  Page 32: (1600, 1131)
  Page 33: (1600, 1131)
  Page 34: (1600, 1131)
  Page 35: (1600, 1131)
  Page 36: (1600, 1131)
  Page 37: (1600, 1131)
  Page 38: (1600, 1131)
  Page 39: (1600, 1131)
  Page 40

### Load model

In [5]:
MODEL_PATH = "zai-org/GLM-OCR"

print("Loading processor and model...")
processor = AutoProcessor.from_pretrained(MODEL_PATH)
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_PATH,
    dtype=torch.bfloat16,
    device_map="auto",
)
model.eval()
print(f"Model loaded on: {next(model.parameters()).device}")


Loading processor and model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/367 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/2.65G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/510 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/165 [00:00<?, ?B/s]

Model loaded on: cuda:0


### Define OCR ingerence function

In [6]:
def run_ocr(image):
    with tempfile.NamedTemporaryFile(suffix=".png", delete=False) as tmp:
        image.save(tmp.name)
        tmp_path = tmp.name

    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "url": tmp_path},
            {"type": "text", "text": "Text Recognition:"},
        ],
    }]

    inputs = processor.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)
    inputs.pop("token_type_ids", None)

    with torch.inference_mode():
        generated_ids = model.generate(**inputs, max_new_tokens=8192)

    output = processor.decode(
        generated_ids[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
    )
    del inputs, generated_ids
    torch.cuda.empty_cache()

    os.unlink(tmp_path)
    return output.strip()


In [8]:
start_time_glm = time.time()

md_pages = []
for i, page_img in enumerate(pages):
    print(f"Page {i + 1}/{len(pages)}...")
    text = run_ocr(page_img)
    md_pages.append(f"# Page {i + 1}\n\n{text}")

md_text = "\n\n".join(md_pages)

# Spara resultat
md_filename = Path(pdf_path).stem + "_GLM_output.md"
with open(md_filename, "w", encoding="utf-8") as f:
    f.write(md_text)

glm_time = time.time() - start_time_glm

print(f"\nGLM-OCR Tid: {glm_time:.2f} sekunder")
print(f"Antal sidor: {len(pages)}")
print(f"\n{md_text[:1000]}")

# Ladda ner resultat
try:
    from google.colab import files
    files.download(md_filename)
except ImportError:
    print(f"Filen sparad lokalt: {md_filename}")

Page 1/68...
Page 2/68...
Page 3/68...
Page 4/68...
Page 5/68...
Page 6/68...
Page 7/68...
Page 8/68...
Page 9/68...
Page 10/68...
Page 11/68...
Page 12/68...
Page 13/68...
Page 14/68...
Page 15/68...
Page 16/68...
Page 17/68...
Page 18/68...
Page 19/68...
Page 20/68...
Page 21/68...
Page 22/68...
Page 23/68...
Page 24/68...
Page 25/68...
Page 26/68...
Page 27/68...
Page 28/68...
Page 29/68...
Page 30/68...
Page 31/68...
Page 32/68...
Page 33/68...
Page 34/68...
Page 35/68...
Page 36/68...
Page 37/68...
Page 38/68...
Page 39/68...
Page 40/68...
Page 41/68...
Page 42/68...
Page 43/68...
Page 44/68...
Page 45/68...
Page 46/68...
Page 47/68...
Page 48/68...
Page 49/68...
Page 50/68...
Page 51/68...
Page 52/68...
Page 53/68...
Page 54/68...
Page 55/68...
Page 56/68...
Page 57/68...
Page 58/68...
Page 59/68...
Page 60/68...
Page 61/68...
Page 62/68...
Page 63/68...
Page 64/68...
Page 65/68...
Page 66/68...
Page 67/68...
Page 68/68...

GLM-OCR Tid: 1769.23 sekunder
Antal sidor: 68

# Page 1


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [9]:
# Visualize page side-by-side with OCR output
import ipywidgets as widgets
from IPython.display import display

page_selector = widgets.IntSlider(
    value=1, min=1, max=len(pages),
    description="Page:",
    continuous_update=False,
    layout=widgets.Layout(width="400px")
)

output = widgets.Output()

def show_page(change):
    idx = page_selector.value - 1
    with output:
        output.clear_output(wait=True)

        # Side-by-side layout
        img_out = widgets.Output(layout=widgets.Layout(width="50%"))
        md_out = widgets.Output(layout=widgets.Layout(width="50%", overflow_y="scroll", height="600px"))

        with img_out:
            display(pages[idx].resize((600, int(pages[idx].height * 600 / pages[idx].width))))

        with md_out:
            print(md_pages[idx])

        display(widgets.HBox([img_out, md_out]))

page_selector.observe(show_page, names="value")
display(page_selector, output)
show_page(None)

IntSlider(value=1, continuous_update=False, description='Page:', layout=Layout(width='400px'), max=68, min=1)

Output()

In [10]:
print(md_pages[34])

# Page 35

Not 7 ÖVRIGA RÖRELSEKOSTNADER 2022 2021
ÖVRIGA RÖRELSEKOSTNADER, KONCERNEN
Kursförlust rörelsen (netto) - 136
Summa övriga rörelsekostnader, koncernen - 136

ÖVRIGA RÖRELSEKOSTNADER, MODERBOLAGET
Kursförlust rörelsen (netto) - -
Summa övriga rörelsekostnader, moderbolaget - -

Not 8 BOKSLUTSDISPOSITIONER 2022 2021
BOKSLUTSDISPOSITIONER, MODERBOLAG
Lämnade koncernbidrag 3 400 1 797
Summa bokslutsdispositioner, moderbolag 3 400 1 797

Not 9 RÄNTEINTÄKTER OCH RÄNTEKOSTNADER 2022 2021
RÄNTEINTÄKTER OCH RÄNTEKOSTNADER, KONCERNEN
Ränteintäkter 24 -
Summa ränteintäkter, koncernen 24 -

Räntekostnader 656 538
Summa räntekostnader, koncernen 656 538

RÄNTEINTÄKTER OCH RÄNTEKOSTNADER, MODERBOLAGET
Ränteintäkter 2 505 1 389
Summa ränteintäkter, moderbolaget 2 505 1 389

Varav koncerninna ränteintäkter 2 505 1 389

Räntekostnader 15 77
Summa räntekostnader, moderbolaget 15 77

Varav koncerninna räntekostnader - -

Not 10 INKOMSTSKATT 2022 2021
INKOMSTSKATT, KONCERNEN
Aktuell skatt -71 -